# Overfitting / generalization checks for verifier-CE

**Check 1 — train-test gap (example memorization).** Score the RELEASED
model on its own training set vs the committed test. Small gap = it
generalized; large gap = it memorized. No retraining (uses the Hub model).

**Check 2 — out-of-distribution (rule vs. generator?).** Train CE on 3
domains (email, payment, repo), test on 2 unseen domains (file, db) whose
actions and resource formats never appeared in training. High OOD accuracy
= learned the rule; a sharp drop = learned the distribution. This one DOES
retrain (the released model saw all domains, so it can't be the OOD model).
budget_violation is payment-only → OOD test covers 8 of 9 classes.

Runtime → T4/L4 GPU → Run all. Download `overfitting_checks.zip` at the end.

In [ ]:
# Idempotent clone + absolute cd (prevents the nested-directory trap).
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
print('cwd:', os.getcwd())
import torch; assert torch.cuda.is_available()
MODEL = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'
!PYTHONPATH=. python test_authority_verifier.py

## Check 1 — train-test gap (released model, no retraining)

In [ ]:
# Rebuild the exact corpus the released model trained on (deterministic),
# then score the released model on TRAIN vs the committed TEST. The gap is
# train_accuracy - test_accuracy.
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 \
  --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25
for name, f in [('TRAIN', 'expanded_train.jsonl'),
                ('COMMITTED_TEST', 'benchmark_test.jsonl')]:
    print(f'\n===== {name} ({f}) =====')
    !PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint {MODEL} --test-file {f}

## Check 2 — out-of-distribution domain hold-out (retrains on 3 domains)

In [ ]:
# Train on email/payment/repo, hold out file/db entirely.
!PYTHONPATH=. python make_ood_split.py --seed 303 --traces-per-class 200 \
  --train-domains email,payment,repo --test-domains file,db
!for s in 7 8 9; do \
  PYTHONPATH=. python train_verifier_reward.py --model Qwen/Qwen2.5-0.5B \
    --ce-loss --balance-reward --prompt-style nl \
    --steps 500 --batch-size 16 --lr 2e-5 \
    --train-file ood_train.jsonl --test-file ood_test.jsonl \
    --eval-every 50 --eval-max-actions 200 --seed $s \
    --log-file training_log_ood_seed$s.jsonl --save-dir ckpt_ood_seed$s || break ; done

In [ ]:
# Each OOD checkpoint scored on the held-out-domain test AND on its own
# training domains — the OOD gap = train-domain acc - held-out-domain acc.
import json
json.dump({'backends': {}}, open('results_ood.json', 'w'))
for s in [7, 8, 9]:
    if not os.path.isdir(f'ckpt_ood_seed{s}'):
        continue
    print(f'\n===== seed {s}: HELD-OUT domains (file,db) =====')
    !PYTHONPATH=. python train_verifier_reward.py \
      --eval-checkpoint ckpt_ood_seed{s} --test-file ood_test.jsonl \
      --merge-results results_ood.json
    print(f'===== seed {s}: TRAIN domains (email,payment,repo) =====')
    !PYTHONPATH=. python train_verifier_reward.py \
      --eval-checkpoint ckpt_ood_seed{s} --test-file ood_train.jsonl

In [ ]:
!zip -j overfitting_checks.zip training_log_ood_seed*.jsonl results_ood.json
from google.colab import files
files.download('overfitting_checks.zip')